In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from nn_utils import split_and_shuffle, evaluate_conv_scalar,train_transformer
import os
from global_vars import *


train_folder = os.path.join(data_folder, 'nodeep_train_data_ver1/')
test_folder = os.path.join(data_folder, 'nodeep_test_data_ver1/')


X_train_val = pd.read_parquet(os.path.join(train_folder, "X_train.parquet"),engine="fastparquet")
y_t_train_val = pd.read_parquet(os.path.join(train_folder, "y_t_train.parquet"),engine="fastparquet")
y_q_train_val = pd.read_parquet(os.path.join(train_folder, "y_q_train.parquet"),engine="fastparquet")

X_test = pd.read_parquet(os.path.join(test_folder, "X_test.parquet"),engine="fastparquet")
y_t_test = pd.read_parquet(os.path.join(test_folder, "y_t_test.parquet"),engine="fastparquet")
y_q_test = pd.read_parquet(os.path.join(test_folder, "y_q_test.parquet"),engine="fastparquet")

X_train, y_train, X_val, y_val = split_and_shuffle(
    X_train_val.values, y_t_train_val.values, val_frac=0.2, seed=0
)


n_surf_features = len(surf_input_var_names)
n_total_features = X_train.shape[1]
branch2_start = (n_total_features-n_surf_features)

Xc_train_flat = X_train[:, range(branch2_start)]
Xs_train = X_train[:, range(branch2_start, n_total_features)]

Xc_val_flat = X_val[:, range(branch2_start)]
Xs_val = X_val[:, range(branch2_start, n_total_features)]

Xc_test_flat = X_test.values[:, range(branch2_start)]
Xs_test = X_test.values[:, range(branch2_start, n_total_features)]

# Unflatten Xc
n_samples_tr = Xc_train_flat.shape[0]
n_samples_va = Xc_val_flat.shape[0]
n_samples_te = Xc_test_flat.shape[0]
n_conv_vars = 4
n1 = 58
Xc_train = Xc_train_flat.reshape(n_samples_tr, n_conv_vars, n1)
Xc_val = Xc_val_flat.reshape(n_samples_va, n_conv_vars, n1)
Xc_test = Xc_test_flat.reshape(n_samples_te, n_conv_vars, n1)



In [2]:
model, scaler, history = train_transformer(
    Xc_train, Xs_train, y_train,
    Xc_val=Xc_val, Xs_val=Xs_val, y_val=y_val,
    model_args={"d_model": 64, "nhead": 4, "num_layers": 3, "dropout": 0.1},
    epochs=200, patience=15, batch_size=256,
    lr=1e-3,                  # this is now the *peak* LR the warmup ramps up to
    use_scheduler=True,
    warmup_frac=0.05,         # first 5% of steps warming up; raise to 0.1 for transformers
    min_lr_ratio=0.05,        # decay bottoms out at 5% of peak lr
)

metrics = evaluate_conv_scalar(model, scaler, Xc_test, Xs_test, y_t_test)
print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

epoch   0 | train 0.8752 | val 0.6234 | lr 1.00e-04
epoch   1 | train 0.6483 | val 0.4961 | lr 2.00e-04
epoch   2 | train 0.5609 | val 0.4588 | lr 3.00e-04
epoch   3 | train 0.5131 | val 0.4404 | lr 4.00e-04
epoch   4 | train 0.4904 | val 0.3756 | lr 5.00e-04
epoch   5 | train 0.4774 | val 0.3805 | lr 6.00e-04
epoch   6 | train 0.4726 | val 0.3687 | lr 7.00e-04
epoch   7 | train 0.4711 | val 0.3667 | lr 8.00e-04
epoch   8 | train 0.4738 | val 0.3710 | lr 9.00e-04
epoch   9 | train 0.4736 | val 0.3712 | lr 1.00e-03
epoch  10 | train 0.4675 | val 0.3608 | lr 1.00e-03
epoch  11 | train 0.4622 | val 0.3567 | lr 1.00e-03
epoch  12 | train 0.4525 | val 0.3523 | lr 9.99e-04
epoch  13 | train 0.4522 | val 0.3386 | lr 9.99e-04
epoch  14 | train 0.4461 | val 0.3660 | lr 9.98e-04
epoch  15 | train 0.4400 | val 0.3308 | lr 9.98e-04
epoch  16 | train 0.4380 | val 0.3308 | lr 9.97e-04
epoch  17 | train 0.4314 | val 0.3268 | lr 9.96e-04
epoch  18 | train 0.4333 | val 0.3536 | lr 9.95e-04
epoch  19 | 

In [3]:
model, scaler, history = train_transformer(
    Xc_train, Xs_train, y_train,
    Xc_val=Xc_val, Xs_val=Xs_val, y_val=y_val,
    model_args={"d_model": 32, "nhead": 4, "num_layers": 2, "dropout": 0.1},
    epochs=200, patience=15, batch_size=256,
    lr=1e-3,                  # this is now the *peak* LR the warmup ramps up to
    use_scheduler=True,
    warmup_frac=0.05,         # first 5% of steps warming up; raise to 0.1 for transformers
    min_lr_ratio=0.05,        # decay bottoms out at 5% of peak lr
)

metrics = evaluate_conv_scalar(model, scaler, Xc_test, Xs_test, y_t_test)
print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

epoch   0 | train 0.8948 | val 0.6487 | lr 1.00e-04
epoch   1 | train 0.6902 | val 0.5390 | lr 2.00e-04
epoch   2 | train 0.6112 | val 0.4752 | lr 3.00e-04
epoch   3 | train 0.5616 | val 0.4307 | lr 4.00e-04
epoch   4 | train 0.5319 | val 0.4282 | lr 5.00e-04
epoch   5 | train 0.5133 | val 0.4052 | lr 6.00e-04
epoch   6 | train 0.5027 | val 0.3966 | lr 7.00e-04
epoch   7 | train 0.4967 | val 0.3933 | lr 8.00e-04
epoch   8 | train 0.4879 | val 0.3802 | lr 9.00e-04
epoch   9 | train 0.4877 | val 0.4496 | lr 1.00e-03
epoch  10 | train 0.4835 | val 0.3696 | lr 1.00e-03
epoch  11 | train 0.4760 | val 0.3806 | lr 1.00e-03
epoch  12 | train 0.4720 | val 0.3612 | lr 9.99e-04
epoch  13 | train 0.4649 | val 0.3553 | lr 9.99e-04
epoch  14 | train 0.4599 | val 0.3576 | lr 9.98e-04
epoch  15 | train 0.4570 | val 0.3417 | lr 9.98e-04
epoch  16 | train 0.4560 | val 0.3447 | lr 9.97e-04
epoch  17 | train 0.4489 | val 0.3717 | lr 9.96e-04
epoch  18 | train 0.4488 | val 0.3338 | lr 9.95e-04
epoch  19 | 

In [4]:
model, scaler, history = train_transformer(
    Xc_train, Xs_train, y_train,
    Xc_val=Xc_val, Xs_val=Xs_val, y_val=y_val,
    model_args={"d_model": 32, "nhead": 4, "num_layers": 4, "dropout": 0.1},
    epochs=200, patience=15, batch_size=256,
    lr=1e-3,                  # this is now the *peak* LR the warmup ramps up to
    use_scheduler=True,
    warmup_frac=0.05,         # first 5% of steps warming up; raise to 0.1 for transformers
    min_lr_ratio=0.05,        # decay bottoms out at 5% of peak lr
)

metrics = evaluate_conv_scalar(model, scaler, Xc_test, Xs_test, y_t_test)
print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

epoch   0 | train 0.8943 | val 0.6625 | lr 1.00e-04
epoch   1 | train 0.6968 | val 0.5507 | lr 2.00e-04
epoch   2 | train 0.6077 | val 0.4726 | lr 3.00e-04
epoch   3 | train 0.5543 | val 0.4220 | lr 4.00e-04
epoch   4 | train 0.5212 | val 0.4048 | lr 5.00e-04
epoch   5 | train 0.4992 | val 0.3921 | lr 6.00e-04
epoch   6 | train 0.4854 | val 0.4537 | lr 7.00e-04
epoch   7 | train 0.4806 | val 0.3598 | lr 8.00e-04
epoch   8 | train 0.4723 | val 0.3876 | lr 9.00e-04
epoch   9 | train 0.4721 | val 0.3747 | lr 1.00e-03
epoch  10 | train 0.4649 | val 0.3620 | lr 1.00e-03
epoch  11 | train 0.4577 | val 0.3339 | lr 1.00e-03
epoch  12 | train 0.4501 | val 0.3536 | lr 9.99e-04
epoch  13 | train 0.4499 | val 0.3661 | lr 9.99e-04
epoch  14 | train 0.4423 | val 0.3584 | lr 9.98e-04
epoch  15 | train 0.4413 | val 0.3304 | lr 9.98e-04
epoch  16 | train 0.4350 | val 0.3418 | lr 9.97e-04
epoch  17 | train 0.4334 | val 0.3455 | lr 9.96e-04
epoch  18 | train 0.4310 | val 0.3288 | lr 9.95e-04
epoch  19 | 

In [5]:
model, scaler, history = train_transformer(
    Xc_train, Xs_train, y_train,
    Xc_val=Xc_val, Xs_val=Xs_val, y_val=y_val,
    model_args={"d_model": 128, "nhead": 4, "num_layers": 3, "dropout": 0.1},
    epochs=200, patience=15, batch_size=256,
    lr=1e-3,                  # this is now the *peak* LR the warmup ramps up to
    use_scheduler=True,
    warmup_frac=0.05,         # first 5% of steps warming up; raise to 0.1 for transformers
    min_lr_ratio=0.05,        # decay bottoms out at 5% of peak lr
)

metrics = evaluate_conv_scalar(model, scaler, Xc_test, Xs_test, y_t_test)
print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

epoch   0 | train 0.8130 | val 0.5583 | lr 1.00e-04
epoch   1 | train 0.5999 | val 0.4608 | lr 2.00e-04
epoch   2 | train 0.5231 | val 0.4522 | lr 3.00e-04
epoch   3 | train 0.4928 | val 0.4006 | lr 4.00e-04
epoch   4 | train 0.4813 | val 0.4270 | lr 5.00e-04
epoch   5 | train 0.4794 | val 0.3762 | lr 6.00e-04
epoch   6 | train 0.4775 | val 0.4056 | lr 7.00e-04
epoch   7 | train 0.4772 | val 0.3975 | lr 8.00e-04
epoch   8 | train 0.4879 | val 0.3895 | lr 9.00e-04
epoch   9 | train 0.4803 | val 0.3690 | lr 1.00e-03
epoch  10 | train 0.4843 | val 0.3611 | lr 1.00e-03
epoch  11 | train 0.4694 | val 0.3674 | lr 1.00e-03
epoch  12 | train 0.4648 | val 0.3485 | lr 9.99e-04
epoch  13 | train 0.4620 | val 0.3801 | lr 9.99e-04
epoch  14 | train 0.4540 | val 0.3517 | lr 9.98e-04
epoch  15 | train 0.4468 | val 0.4098 | lr 9.98e-04
epoch  16 | train 0.4478 | val 0.3710 | lr 9.97e-04
epoch  17 | train 0.4447 | val 0.3694 | lr 9.96e-04
epoch  18 | train 0.4399 | val 0.3479 | lr 9.95e-04
epoch  19 | 

In [6]:
model, scaler, history = train_transformer(
    Xc_train, Xs_train, y_train,
    Xc_val=Xc_val, Xs_val=Xs_val, y_val=y_val,
    model_args={"d_model": 64, "nhead": 8, "num_layers": 3, "dropout": 0.1},
    epochs=200, patience=15, batch_size=256,
    lr=1e-3,                  # this is now the *peak* LR the warmup ramps up to
    use_scheduler=True,
    warmup_frac=0.05,         # first 5% of steps warming up; raise to 0.1 for transformers
    min_lr_ratio=0.05,        # decay bottoms out at 5% of peak lr
)

metrics = evaluate_conv_scalar(model, scaler, Xc_test, Xs_test, y_t_test)
print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

epoch   0 | train 0.8909 | val 0.6160 | lr 1.00e-04
epoch   1 | train 0.6398 | val 0.4757 | lr 2.00e-04
epoch   2 | train 0.5513 | val 0.4255 | lr 3.00e-04
epoch   3 | train 0.5100 | val 0.3948 | lr 4.00e-04
epoch   4 | train 0.4819 | val 0.3794 | lr 5.00e-04
epoch   5 | train 0.4722 | val 0.3681 | lr 6.00e-04
epoch   6 | train 0.4656 | val 0.3650 | lr 7.00e-04
epoch   7 | train 0.4628 | val 0.3604 | lr 8.00e-04
epoch   8 | train 0.4605 | val 0.3477 | lr 9.00e-04
epoch   9 | train 0.4617 | val 0.3448 | lr 1.00e-03
epoch  10 | train 0.4585 | val 0.3379 | lr 1.00e-03
epoch  11 | train 0.4546 | val 0.3838 | lr 1.00e-03
epoch  12 | train 0.4433 | val 0.4216 | lr 9.99e-04
epoch  13 | train 0.4392 | val 0.3850 | lr 9.99e-04
epoch  14 | train 0.4386 | val 0.3274 | lr 9.98e-04
epoch  15 | train 0.4312 | val 0.3260 | lr 9.98e-04
epoch  16 | train 0.4279 | val 0.3641 | lr 9.97e-04
epoch  17 | train 0.4263 | val 0.3196 | lr 9.96e-04
epoch  18 | train 0.4248 | val 0.3095 | lr 9.95e-04
epoch  19 | 

In [2]:
model, scaler, history = train_transformer(
    Xc_train, Xs_train, y_train,
    Xc_val=Xc_val, Xs_val=Xs_val, y_val=y_val,
    model_args={"d_model": 16, "nhead": 8, "num_layers": 4, "dropout": 0.1},
    epochs=200, patience=15, batch_size=256,
    lr=1e-3,                  # this is now the *peak* LR the warmup ramps up to
    use_scheduler=True,
    warmup_frac=0.05,         # first 5% of steps warming up; raise to 0.1 for transformers
    min_lr_ratio=0.05,        # decay bottoms out at 5% of peak lr
)

metrics = evaluate_conv_scalar(model, scaler, Xc_test, Xs_test, y_t_test)
print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

epoch   0 | train 0.9274 | val 0.6982 | lr 1.00e-04
epoch   1 | train 0.7192 | val 0.5597 | lr 2.00e-04
epoch   2 | train 0.6284 | val 0.4886 | lr 3.00e-04
epoch   3 | train 0.5806 | val 0.4728 | lr 4.00e-04
epoch   4 | train 0.5482 | val 0.4260 | lr 5.00e-04
epoch   5 | train 0.5251 | val 0.4431 | lr 6.00e-04
epoch   6 | train 0.5103 | val 0.3899 | lr 7.00e-04
epoch   7 | train 0.4977 | val 0.3882 | lr 8.00e-04
epoch   8 | train 0.4889 | val 0.3779 | lr 9.00e-04
epoch   9 | train 0.4825 | val 0.3692 | lr 1.00e-03
epoch  10 | train 0.4779 | val 0.3718 | lr 1.00e-03
epoch  11 | train 0.4672 | val 0.3497 | lr 1.00e-03
epoch  12 | train 0.4646 | val 0.3605 | lr 9.99e-04
epoch  13 | train 0.4610 | val 0.3541 | lr 9.99e-04
epoch  14 | train 0.4536 | val 0.3861 | lr 9.98e-04
epoch  15 | train 0.4520 | val 0.3566 | lr 9.98e-04
epoch  16 | train 0.4478 | val 0.3373 | lr 9.97e-04
epoch  17 | train 0.4438 | val 0.3519 | lr 9.96e-04
epoch  18 | train 0.4417 | val 0.3404 | lr 9.95e-04
epoch  19 | 

In [3]:
model, scaler, history = train_transformer(
    Xc_train, Xs_train, y_train,
    Xc_val=Xc_val, Xs_val=Xs_val, y_val=y_val,
    model_args={"d_model": 16, "nhead": 8, "num_layers": 5, "dropout": 0.1},
    epochs=200, patience=15, batch_size=256,
    lr=1e-3,                  # this is now the *peak* LR the warmup ramps up to
    use_scheduler=True,
    warmup_frac=0.05,         # first 5% of steps warming up; raise to 0.1 for transformers
    min_lr_ratio=0.05,        # decay bottoms out at 5% of peak lr
)

metrics = evaluate_conv_scalar(model, scaler, Xc_test, Xs_test, y_t_test)
print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

epoch   0 | train 0.9588 | val 0.7305 | lr 1.00e-04
epoch   1 | train 0.7421 | val 0.5796 | lr 2.00e-04
epoch   2 | train 0.6378 | val 0.5073 | lr 3.00e-04
epoch   3 | train 0.5803 | val 0.4938 | lr 4.00e-04
epoch   4 | train 0.5424 | val 0.4216 | lr 5.00e-04
epoch   5 | train 0.5205 | val 0.4211 | lr 6.00e-04
epoch   6 | train 0.5064 | val 0.5747 | lr 7.00e-04
epoch   7 | train 0.4971 | val 0.4303 | lr 8.00e-04
epoch   8 | train 0.4879 | val 0.3802 | lr 9.00e-04
epoch   9 | train 0.4835 | val 0.3921 | lr 1.00e-03
epoch  10 | train 0.4775 | val 0.3757 | lr 1.00e-03
epoch  11 | train 0.4695 | val 0.3844 | lr 1.00e-03
epoch  12 | train 0.4641 | val 0.3708 | lr 9.99e-04
epoch  13 | train 0.4565 | val 0.3364 | lr 9.99e-04
epoch  14 | train 0.4527 | val 0.3718 | lr 9.98e-04
epoch  15 | train 0.4509 | val 0.3476 | lr 9.98e-04
epoch  16 | train 0.4477 | val 0.3364 | lr 9.97e-04
epoch  17 | train 0.4424 | val 0.3395 | lr 9.96e-04
epoch  18 | train 0.4410 | val 0.3404 | lr 9.95e-04
epoch  19 | 

In [4]:
model, scaler, history = train_transformer(
    Xc_train, Xs_train, y_train,
    Xc_val=Xc_val, Xs_val=Xs_val, y_val=y_val,
    model_args={"d_model": 32, "nhead": 8, "num_layers": 5, "dropout": 0.1},
    epochs=200, patience=15, batch_size=256,
    lr=1e-3,                  # this is now the *peak* LR the warmup ramps up to
    use_scheduler=True,
    warmup_frac=0.05,         # first 5% of steps warming up; raise to 0.1 for transformers
    min_lr_ratio=0.05,        # decay bottoms out at 5% of peak lr
)

metrics = evaluate_conv_scalar(model, scaler, Xc_test, Xs_test, y_t_test)
print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

epoch   0 | train 0.8984 | val 0.6647 | lr 1.00e-04
epoch   1 | train 0.6917 | val 0.5218 | lr 2.00e-04
epoch   2 | train 0.5946 | val 0.4680 | lr 3.00e-04
epoch   3 | train 0.5431 | val 0.4377 | lr 4.00e-04
epoch   4 | train 0.5102 | val 0.3888 | lr 5.00e-04
epoch   5 | train 0.4902 | val 0.3791 | lr 6.00e-04
epoch   6 | train 0.4767 | val 0.3967 | lr 7.00e-04
epoch   7 | train 0.4722 | val 0.3638 | lr 8.00e-04
epoch   8 | train 0.4654 | val 0.3735 | lr 9.00e-04
epoch   9 | train 0.4710 | val 0.3850 | lr 1.00e-03
epoch  10 | train 0.4598 | val 0.4379 | lr 1.00e-03
epoch  11 | train 0.4521 | val 0.3349 | lr 1.00e-03
epoch  12 | train 0.4475 | val 0.3361 | lr 9.99e-04
epoch  13 | train 0.4425 | val 0.3362 | lr 9.99e-04
epoch  14 | train 0.4367 | val 0.3296 | lr 9.98e-04
epoch  15 | train 0.4320 | val 0.3344 | lr 9.98e-04
epoch  16 | train 0.4314 | val 0.3673 | lr 9.97e-04
epoch  17 | train 0.4260 | val 0.3237 | lr 9.96e-04
epoch  18 | train 0.4228 | val 0.3374 | lr 9.95e-04
epoch  19 | 

In [ ]:
lst_d = [8,16,32,64]

lst_nl = [3,4,5,6]

lst_nh = [2,4,8,16]


for d in lst_d:
    for nl in lst_nl:
        for nh in lst_nh:
            print(d,nl,nh)

            model, scaler, history = train_transformer(
                Xc_train, Xs_train, y_train,
                Xc_val=Xc_val, Xs_val=Xs_val, y_val=y_val,
                model_args={"d_model": d, "nhead": nh, "num_layers": nl, "dropout": 0.1},
                epochs=250, patience=20, batch_size=256,
                lr=1e-3,                  # this is now the *peak* LR the warmup ramps up to
                use_scheduler=True,
                warmup_frac=0.05,         # first 5% of steps warming up; raise to 0.1 for transformers
                min_lr_ratio=0.05,        # decay bottoms out at 5% of peak lr
                verbose=False
            )
            
            metrics = evaluate_conv_scalar(model, scaler, Xc_test, Xs_test, y_t_test)
            print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

8 3 2
